# Sprint 3 — 正式篩選前驗證

對 Sprint 1/2 產出的 Parquet 進行品質與樣本量驗證，確認後續分析資料可用。

> ⚠️ **本 Notebook 純屬驗證用途，不產生任何輸出檔案，不需重跑。**

## Cell 1 — 引入資料

設定四個路徑：`Member.csv`、`Order_TG.csv`、`dormant_members.parquet`、`dormant_activity_merged.parquet`。

> ℹ️ 不產生輸出檔案

In [ ]:
import duckdb

con = duckdb.connect()

MEMBER_PATH  = '91APP_Dataset(main)/Member.csv'
ORDER_PATH   = '91APP_Dataset(main)/Order_TG.csv'
DORMANT_PATH = 'output/sprint1/dormant_members.parquet'
MERGED_PATH  = 'output/sprint2/dormant_activity_merged.parquet'

print('路徑設定完成')

## Cell 2 — 漏斗數字

計算並印出四層漏斗數字：全體會員 → 有購買記錄 → 沉睡客 → 有回流沉睡客，確認各層比例合理。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**

In [ ]:
# 全體會員
total_members = con.execute(f"""
    SELECT COUNT(DISTINCT ShopMemberId) FROM read_csv_auto('{MEMBER_PATH}')
""").fetchone()[0]

# 有購買記錄的會員（StatusDef=Finish, < 2023-09-01）
buyers = con.execute(f"""
    SELECT COUNT(DISTINCT ShopMemberId)
    FROM read_csv_auto('{ORDER_PATH}')
    WHERE StatusDef = 'Finish' AND OrderDateTime < '2023-09-01'
""").fetchone()[0]

# 沉睡客
dormant = con.execute(f"SELECT COUNT(*) FROM read_parquet('{DORMANT_PATH}')").fetchone()[0]

# 有回流的沉睡客
reactivated = con.execute(f"SELECT COUNT(*) FROM read_parquet('{MERGED_PATH}')").fetchone()[0]

print(f"{'全體會員':<20} {total_members:>10,}")
print(f"{'有購買記錄':<20} {buyers:>10,}  ({buyers/total_members*100:.1f}% of 全體)")
print(f"{'沉睡客':<20} {dormant:>10,}  ({dormant/buyers*100:.1f}% of 有購買記錄)")
print(f"{'有回流沉睡客':<20} {reactivated:>10,}  ({reactivated/dormant*100:.1f}% of 沉睡客)")

## Cell 3 — 樣本量檢查

確認 `dormant_activity_merged` 的筆數是否達到分析門檻（≥ 1,000 筆）。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**

In [ ]:
THRESHOLD = 1_000
result = 'PASS' if reactivated >= THRESHOLD else 'FAIL'
print(f"dormant_activity_merged 筆數 : {reactivated:,}")
print(f"門檻（>= {THRESHOLD:,}）          : {result}")

## Cell 4 — dormant_activity_merged 基本分布

確認 `t0` 的最早/最晚日期、`total_events` 與 `active_days` 的中位數與最大值，驗證資料分布符合預期。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**

In [ ]:
dist = con.execute(f"""
SELECT
    MIN(t0)::DATE           AS t0_min,
    MAX(t0)::DATE           AS t0_max,
    MEDIAN(total_events)    AS med_events,
    MAX(total_events)       AS max_events,
    MEDIAN(active_days)     AS med_days,
    MAX(active_days)        AS max_days
FROM read_parquet('{MERGED_PATH}')
""").fetchone()

print(f"t0 最早日期         : {dist[0]}")
print(f"t0 最晚日期         : {dist[1]}")
print(f"total_events 中位數 / 最大 : {dist[2]} / {dist[3]:,}")
print(f"active_days  中位數 / 最大 : {dist[4]} / {dist[5]:,}")

## Cell 5 — 缺值率檢查

對 `dormant_members.parquet` 與 `dormant_activity_merged.parquet` 的所有欄位逐一計算缺值數與缺值率，標記有缺值的欄位。

> ℹ️ **純統計用，不產生輸出檔案，不需重跑。**

In [ ]:
for label, path in [('dormant_members', DORMANT_PATH), ('dormant_activity_merged', MERGED_PATH)]:
    cols = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]

    print(f"=== {label}（共 {total:,} 筆）===")
    print(f"  {'欄位':<28} {'缺值數':>8} {'缺值率':>8}")
    print(f"  {'-'*48}")
    any_null = False
    for col in cols['column_name']:
        null_cnt = con.execute(f"""
            SELECT COALESCE(SUM(CASE WHEN \"{col}\" IS NULL THEN 1 END), 0)
            FROM read_parquet('{path}')
        """).fetchone()[0]
        pct = null_cnt / total * 100
        flag = ' <-- WARNING' if pct > 0 else ''
        print(f"  {col:<28} {null_cnt:>8,} {pct:>7.2f}%{flag}")
        if pct > 0:
            any_null = True
    print(f"  {'OK - 無缺值' if not any_null else '有缺值欄位，請確認'}")
    print()